# Week 96, Day 6: Review

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w096d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import ast
old = "def f(x):\n    '''Old.'''\n    return x\n"
new = "def f(x):\n    '''New words.'''\n    return x\n"
strip = lambda s: ast.dump(ast.parse(s).body[0].body[1])
print(strip(old) == strip(new))

**Predict before you run:** what will this print?



In [ ]:
import csv
print(next(csv.reader(['a,"b,c",'])))

**Predict before you run:** what will this print?



In [ ]:
import itertools
print(sum(1 for n in range(3) for _ in itertools.product("ab,", repeat=n)))

**Predict before you run:** what will this print?



In [ ]:
import re
print(re.sub(r"\bgh-(\d+)", r"issue \1", "fixes gh-77 and gh-80"))

**Predict before you run:** what will this print?



In [ ]:
labels = ["issue (blocking)", "nitpick", "praise"]
print("request changes" if any("(blocking)" in l for l in labels) else "approve")

### Exercise: Reading a review comment

Write `parse_comment(text)`: for strings like `"Lib/wraptext.py:34: issue (blocking): a bare except"` or `"praise: thanks!"`, return a dict with `path` and `line` (an int) or `None` for both, `label`, `decoration` (or `None`) and `subject`. Raise `ValueError("not a conventional comment")` otherwise.

In [ ]:
import re

def parse_comment(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_parse():\n    \"back into parts\"\n    assert parse_comment(\"Lib/wraptext.py:34: issue (blocking): a bare except\") == {\n        \"path\": \"Lib/wraptext.py\", \"line\": 34, \"label\": \"issue\", \"decoration\": \"blocking\", \"subject\": \"a bare except\"}\n    assert parse_comment(\"praise: thanks!\") == {\"path\": None, \"line\": None, \"label\": \"praise\", \"decoration\": None, \"subject\": \"thanks!\"}\n    try:\n        parse_comment(\"This is wrong.\")\n    except ValueError as e:\n        assert str(e) == \"not a conventional comment\"\n    else:\n        raise AssertionError(\"plain text\")")

### Exercise: Exhaustively, against an oracle

Write `exhaustive(fn, oracle, alphabet, max_len)`: every string over `alphabet` of length 0 to `max_len` (in order of length, then `itertools.product` order) where `fn` and `oracle` disagree.

In [ ]:
import itertools

def exhaustive(fn, oracle, alphabet, max_len):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_exhaustive():\n    \"every short input\"\n    upper_first = lambda s: s[:1].upper() + s[1:]\n    assert exhaustive(upper_first, str.capitalize, \"aB\", 2) == [\"aB\", \"BB\"]\n    assert exhaustive(str.lower, str.lower, \"ab\", 3) == []")

### Exercise: The sprint board

Write `sprint_board(entries, today)`. Each entry has `pr`, `title`, `status` (`"open"`, `"merged"` or `"closed"`), `ci` (`"passed"`, `"failed"` or `"pending"`), `waiting_on` (`"me"`, `"reviewer"` or `None`), `conflict` (a bool) and `last_activity` (`"YYYY-MM-DD"`). For **open** entries, actions in priority order:

1. `"fix CI: #PR TITLE"` if CI failed;
2. `"resolve conflict: #PR TITLE"` if there's a conflict;
3. `"reply: #PR TITLE"` if it's waiting on you;
4. `"polite ping: #PR TITLE (D days)"` if it's waiting on the reviewer and the last activity was at least 14 days before today.

An entry can need several actions; list all actions of priority 1 first (by PR number), then priority 2, and so on. Return `(lines, next_step)`: `lines` is the actions followed by `"open: O, merged: M, closed: C"`; `next_step` is `"next: "` and the first action, or `"next: start something new"`.

In [ ]:
from datetime import date

def sprint_board(entries, today):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def e(pr, title, status=\"open\", ci=\"passed\", waiting_on=None, conflict=False, last=\"2026-12-01\"):\n    return {\"pr\": pr, \"title\": title, \"status\": status, \"ci\": ci, \"waiting_on\": waiting_on, \"conflict\": conflict, \"last_activity\": last}\n\nENTRIES = [\n    e(140210, \"Clarify json.dumps sort_keys\", status=\"merged\"),\n    e(140188, \"Keep a word that exactly fills the width\", waiting_on=\"reviewer\", last=\"2026-11-20\"),\n    e(2815, \"Fix a broken example\", ci=\"failed\", waiting_on=\"me\", last=\"2026-12-08\"),\n    e(140240, \"Test calendar.monthrange\", conflict=True, waiting_on=\"reviewer\", last=\"2026-12-02\"),\n    e(140199, \"Clarify difflib cutoff\", status=\"closed\"),\n    e(140250, \"Document shutil.copy2 times\", waiting_on=\"me\", last=\"2026-12-09\"),\n]\n\ndef test_board():\n    \"this morning's list\"\n    lines, next_step = sprint_board(ENTRIES, \"2026-12-10\")\n    assert lines == [\n        \"fix CI: #2815 Fix a broken example\",\n        \"resolve conflict: #140240 Test calendar.monthrange\",\n        \"reply: #2815 Fix a broken example\",\n        \"reply: #140250 Document shutil.copy2 times\",\n        \"polite ping: #140188 Keep a word that exactly fills the width (20 days)\",\n        \"open: 4, merged: 1, closed: 1\",\n    ]\n    assert next_step == \"next: fix CI: #2815 Fix a broken example\"\n\ndef test_quiet():\n    \"nothing waiting on you\"\n    lines, next_step = sprint_board([e(1, \"A\", waiting_on=\"reviewer\", last=\"2026-12-01\")], \"2026-12-10\")\n    assert lines == [\"open: 1, merged: 0, closed: 0\"] and next_step == \"next: start something new\"\n    assert sprint_board([e(1, \"A\", waiting_on=\"reviewer\", last=\"2026-11-26\")], \"2026-12-10\")[0][0] == \"polite ping: #1 A (14 days)\"")

> **Three weeks of open source.** You started by reading a codebase you didn't write and finished by reviewing other people's changes to it. In between, the code was rarely the hard part: finding the right issue, reproducing it exactly, making the change small, writing the test, answering review with patience. Those are the same skills a team uses every day. What surprised you most about how open-source projects work?

### Exercise: Bonus: choosing a pull request to review

Write `review_queue(prs, today, me)`: each pull request has `pr`, `author`, `title`, `reviews` (a count), `has_test` (a bool), `lines` (changed lines), `labels` and `opened`. Leave out your own (`author == me`) and any with `do-not-merge` in its labels. Score the rest: 3 for no reviews yet, 1 if it has a test, 2 if under 50 changed lines (1 if under 200), 1 for a `docs` or `tests` label, minus 1 if opened more than 60 days before today. Return `"#PR (SCORE): TITLE"` for the top `n=3`, highest first, ties by older (smaller number) first.

In [ ]:
from datetime import date

def review_queue(prs, today, me, n=3):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def p(pr, title, author=\"someone\", reviews=0, has_test=True, lines=30, labels=(), opened=\"2026-12-01\"):\n    return {\"pr\": pr, \"author\": author, \"title\": title, \"reviews\": reviews, \"has_test\": has_test, \"lines\": lines,\n            \"labels\": list(labels), \"opened\": opened}\n\nPRS = [\n    p(140260, \"Fix difflib cutoff=0\"),\n    p(140261, \"Rewrite the tokenizer\", lines=4200, has_test=False),\n    p(140262, \"Clarify json docs\", labels=[\"docs\"], has_test=False, lines=8),\n    p(140263, \"My own fix\", author=\"ada\"),\n    p(140264, \"Add calendar tests\", labels=[\"tests\"], reviews=2, lines=120),\n    p(139001, \"Old small fix\", opened=\"2026-08-01\", lines=12),\n    p(140265, \"Experiment\", labels=[\"do-not-merge\"]),\n]\n\ndef test_queue():\n    \"where a review helps most\"\n    assert review_queue(PRS, \"2026-12-10\", \"ada\") == [\"#140260 (6): Fix difflib cutoff=0\", \"#140262 (6): Clarify json docs\",\n                                                      \"#139001 (5): Old small fix\"]\n    assert len(review_queue(PRS, \"2026-12-10\", \"ada\", n=10)) == 5")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

## Mixed review

## Several at once

Open source runs on other people's time, so a contributor usually has several things in flight: a pull request waiting for review, another with a CI failure, a third with a question to answer, a review promised to someone. The trick is a short daily routine with a clear order: first anything **broken** (a failing CI run, a conflict with `main`), then anything **waiting on you** (a reviewer's comment), then anything that's been **waiting on others** for a long time (a single polite note after two weeks or so: "Is there anything else needed here?"), then new work. Closed pull requests get a line in the log and, if the reason is useful, in the write-up.

## Bonus: what to review next

## Weekly quiz

## Week 96 review

Tick off each one honestly. If one feels shaky, go back to that day.

- I can find drifted examples, signatures and references, prove a change is docs only, and correct examples. ([Day 1](w096d1.html))
- I can measure line coverage with sys.settrace, find untested statements and fill them with checking tests. ([Day 2](w096d2.html))
- I can fix a bug at sprint speed with an oracle, shrinking, seeded fuzzing and exhaustive checks. ([Day 3](w096d3.html))
- I can review a pull request against a checklist with labelled, kind comments and a clear verdict. ([Day 4](w096d4.html))
- I can keep a contribution log and turn it into honest numbers, links, a CV line and a post. ([Day 5](w096d5.html))
- I can run a daily board of my open work and choose pull requests to review. (Today)

## Your turn

1. **Run your sprint board** on your real open work, and do the first action.
2. **Choose two pull requests to review** this week with your `review_queue`.
3. **Finish and publish** your write-up, linked from your portfolio.

Tomorrow is a **rest day**. Next week begins **Interview mastery I**: timed problem sets, a system design round, a code review interview, behavioural questions, and a full mock loop.